# Suite PYPRS — Parsing Python

The versions of Python parse source text into Python trees by delegating to tree-sitter-python, then rewriting its
concrete syntax tree into Python kinds. The converter regroups what tree-sitter-python 0.25 groups unlike Python, and
a pre-pass handles the constructs of Python 3.13 and later that it cannot parse. Each case parses constructs and
checks the tree (by shape where it matters, and by the text it prints), then the errors, each with its line and
column. In Python, CPython's own parser checks the conformance source too.

In [ ]:
import { readFileSync } from "node:fs";

import { Errors, Syntax as F } from "@mbse/programs/Framework";
import { Python312, Python314, Syntax as S } from "@mbse/programs/Python";
import * as Parser from "@mbse/programs/Python/_Parser";
import { PythonStandard } from "@mbse/programs/Python/_Standard";

import { assert, equal, raises } from "./support.js";

const { ParseError } = Errors;
const PY315 = new PythonStandard(3, 15);
type Standard = { parse(text: string): S.Module; print(node: F.SyntaxNode): string };

function parse(text: string, standard: Standard = PY315): S.Module {
  const module = standard.parse(text);
  equal(S.LANGUAGE.validate(module), []);
  return module;
}

const first = (text: string, standard: Standard = PY315): any => parse(text, standard).body[0];
/** The expression of the expression statement `text`. */
const value = (text: string): any => first(text).value;

/** Parses `text`, checks it prints `expected` (by default itself) and that the print parses to the same tree. */
function sameText(text: string, expected: string | null = null, standard: Standard = PY315): S.Module {
  const module = parse(text, standard);
  const printed = standard.print(module);
  assert(printed === (expected ?? text) + "\n", printed);
  assert(F.same(parse(printed, standard), module));
  return module;
}

function fails(text: string, message: string, line: number, column: number, standard: Standard = PY315): void {
  raises(ParseError, () => standard.parse(text), message);
  try {
    standard.parse(text);
  } catch (error) {
    const e = error as InstanceType<typeof ParseError>;
    equal([e.line, e.column], [line, column]);
  }
}

const kinds = (nodes: readonly F.SyntaxNode[]) => nodes.map((n) => n.kind().KIND);

## PYPRS-01 · The conformance source parses to a valid tree, which prints and parses back the same, and means what CPython reads

In [ ]:
const text = readFileSync("../../conformance/sources/python314.py", "utf8");
const corpus = parse(text, Python314);
const printed = Python314.print(corpus);
assert(F.same(parse(printed, Python314), corpus) && Python314.print(parse(printed, Python314)) === printed);
equal(new Set([...F.walk(corpus)].map((n) => n.kind().KIND)), new Set(S.LANGUAGE.kinds().keys()));
const [parsed, positions] = Parser.parse(text);
assert([...F.walk(parsed)].every((n) => positions.has(n))); // every node is placed, for the version's problems
console.log(printed.split("\n").length - 1, "lines");

## PYPRS-02 · Literals keep their spelling: numbers, strings and bytes, prefixes and quotes, and concatenation

In [ ]:
for (const literal of ["0", "1_000", "0x_FF", "0o17", "0b1010", "1.5e-3", "2j", "3.0J", ".5", "5.", "None", "True", "False",
  "...", "'a'", "\"a\"", "'''a'''", "\"\"\"a\nb\"\"\"", "b'a'", "rb'\\d'", "R'a'", "u'a'", "'\\n\\u00e9'"]) {
  assert(F.same(value(literal), new S.Constant({ spelling: literal })));
  sameText(literal);
}
equal(kinds(value("'a' \"b\" f'{c}'").values), ["Constant", "Constant", "JoinedStr"]);
sameText("('a'\n 'b')", "('a' 'b')"); // layout is the printer's
console.log("ok");

## PYPRS-03 · f-strings and t-strings: their text, replacement fields, conversions, format specs and self-documenting fields

In [ ]:
const joined = value("f'a{b!r:>{w}} {c = } {{d}}'");
equal([joined.prefix, joined.quote], ["f", "'"]);
equal(kinds(joined.values), ["StringText", "FormattedValue", "StringText", "FormattedValue", "StringText"]);
const field = joined.values[1];
equal([field.text, field.conversion, field.debug], ["b", "r", false]);
equal(kinds(field.format_spec.values), ["StringText", "FormattedValue"]);
assert(field.format_spec.values[0].spelling === ">" && field.format_spec.values[1].text === "w");
equal([joined.values[3].text, joined.values[3].debug], ["c = ", true]);
assert(joined.values[4].spelling === " {{d}}");
const template = value("t'{x!s:{y}}'");
assert(template instanceof S.TemplateStr && template.values[0] instanceof S.Interpolation);
assert(template.values[0].format_spec.values[0] instanceof S.FormattedValue);
assert(value("rf'\\\\'").values[0].spelling === "\\\\"); // which tree-sitter-python counts as part of the quote
assert(value("f\"{x:=10}\"").values[0].format_spec.values[0].spelling === "=10"); // a spec, not an assignment
assert(value("f\"{(x:=10)}\"").values[0].value instanceof S.Parenthesized);
assert(value("F\"\"\"\n{x}\n\"\"\"").quote === "\"\"\"" && value("Rf''").values.length === 0);
for (const text of ["f'a{b!r:>{w}} {c = } {{d}}'", "t'{x!s:{y}}'", "rf'\\\\'", "f\"{x:=10}\"", "f\"{x = !r:^10}\"",
  "f'{\"nested\"}'", "f\"{f'{x}'}\"", "f'{x,y}'", "f'{a, *b}'", "f'{yield x}'"]) {
  sameText(text);
}
console.log("ok");

## PYPRS-04 · Expressions, with Python's precedence where tree-sitter-python's differs

In [ ]:
const regrouped = value("~a ^ b & c | d & e ^ -f"); // `&` binds tighter than `^`
assert(Python314.print(regrouped) === "~a ^ b & c | d & e ^ -f");
equal([regrouped.op, regrouped.left.op, regrouped.left.right.op], ["|", "^", "&"]);
assert(value("a ** b ** c").right.op === "**"); // `**` groups to the right
assert(value("a - b - c").left.op === "-");
const awaited = value("await x ** 2");
assert(awaited instanceof S.BinOp && awaited.left instanceof S.Await); // `(await x) ** 2`
const walrus = value("(m := a if b else c)").value;
assert(walrus instanceof S.NamedExpr && walrus.value instanceof S.IfExp);
equal(kinds(value("a < b <= c is not d not in e").comparisons), ["Comparison", "Comparison", "Comparison", "Comparison"]);
equal(value("a is not b not in c").comparisons.map((c: S.Comparison) => c.op), ["is not", "not in"]);
equal(value("a is  not b").comparisons.map((c: S.Comparison) => c.op), ["is not"]);
assert(value("a and b and c").values.length === 3 && value("a or b and c").values.length === 2);
assert(value("a and (b and c)").values.length === 2);
assert(value("not a") instanceof S.UnaryOp && value("-a").op === "-");
const call = value("f(a, *b, c=1, **d)");
equal(kinds(call.args), ["Name", "Starred"]);
assert(call.keywords[1].arg === null);
equal(kinds(value("f(x for x in y)").args), ["GeneratorExp"]);
equal(kinds(value("a[1:2, ::3, *s]").slice.elts), ["Slice", "Slice", "Starred"]);
assert(value("a[1,]").slice.elts[0].spelling === "1" && value("a[1]").slice instanceof S.Constant);
assert(F.same(value("a[::]").slice, new S.Slice()));
assert(value("()") instanceof S.Tuple && value("(1,)") instanceof S.Parenthesized);
equal(kinds(value("1, 2").elts), ["Constant", "Constant"]);
equal(kinds(value("f(),").elts), ["Call"]);
assert(value("{**a, 'b': 1}").items[0].key === null);
for (const text of ["a + b * c", "(a + b) * c", "-a ** b", "(-a) ** b", "a if b else c if d else e", "lambda: 0",
  "lambda a, /, b=1, *c, d, **e: a", "[x for x in y if x if not x]", "{k: v for k, v in d}",
  "{x for x in y}", "(x for x in y)", "[x async for x in y]", "[*a, *b]", "{1, *a}", "{**a, 'b': 1}",
  "a.b.c", "a[b][c]", "f(a)(b)", "await a", "yield", "yield a, b", "x = yield", "(yield from a)",
  "a[b:c]", "a[:]", "a[b:]", "a[:c]", "a[::d]", "f(*(a for a in b))", "print >> f, a", "print >> f"]) {
  sameText(text);
}
sameText("f((x for x in y))", "f(x for x in y)"); // the call's parentheses are the generator's
const python2 = value("[x for x in 1, 2]"); // Python 2's iterable, which tree-sitter-python reads, prints as Python 3's
assert(python2.generators[0].iter instanceof S.Tuple && PY315.print(python2) === "[x for x in (1, 2)]");
console.log("ok");

## PYPRS-05 · Assignment targets, and annotated and augmented assignment

In [ ]:
equal(kinds(first("a = b = c").targets), ["Name", "Name"]);
equal(kinds(first("a, *b = c").targets[0].elts), ["Name", "Starred"]);
assert(first("[a, b] = c").targets[0] instanceof S.List);
assert(first("(a, b) = c").targets[0].value instanceof S.Tuple && first("(a) = c").targets[0] instanceof S.Parenthesized);
assert(F.same(first("() = c").targets[0], new S.Tuple()));
assert(first("x += 1").op === "+" && first("x //= 1").op === "//" && first("x **= 1").op === "**");
const annotated = first("x: int = 1");
assert(annotated instanceof S.AnnAssign && annotated.value !== null && first("x: int").value === null);
assert(first("a.b: int").target instanceof S.Attribute);
const renamed = first("type(t).__name__ = 'x'"); // an assignment, which tree-sitter-python reads as a type alias
assert(renamed instanceof S.Assign && renamed.targets[0].value.func.id.spelling === "type");
const union = first("def f(a: list[str] | str | None): pass").args.args[0].annotation;
assert(union.left.op === "|" && union.left.left instanceof S.Subscript); // `|` groups to the left
for (const text of ["a = b = c", "a, *b = c", "[a, b] = c", "(a, b) = c", "(a,) = c", "() = c", "a.b[c] = d", "x @= 1",
  "x: list[int] | None = None", "x: int", "a = yield b", "for (a, b), c in d:\n    pass",
  "type(t).__name__ = 'x'", "type(m)[0] = p", "type[0] = 1"]) {
  sameText(text);
}
console.log("ok");

## PYPRS-06 · Simple statements: imports, global and nonlocal, del, assert, raise, return and type aliases

In [ ]:
const imported = first("import a.b.c as d, e");
equal(imported.names.map((a: S.Alias) => (a.name as S.DottedName).names.at(-1)?.spelling), ["c", "e"]);
assert(imported.names[0].asname.spelling === "d");
const relative = first("from ..m import x as y, z");
equal([relative.level, relative.module.names[0].spelling, relative.names.length], [2n, "m", 2]);
assert(first("from . import x").module === null && first("from . import x").level === 1n);
assert(first("from m import *").names[0].name === null && first("from m import x").level === null);
const future = first("from __future__ import annotations");
assert(future.module.names[0].spelling === "__future__" && future.names[0].name.names[0].spelling === "annotations");
assert(first("from ... import x").level === 3n);
const alias = first("type A[T] = list[T]");
assert(alias instanceof S.TypeAlias && alias.name.id.spelling === "A" && alias.value instanceof S.Subscript);
for (const text of ["import a.b.c as d, e", "from ..m import x as y, z", "from . import x", "from m import *",
  "from __future__ import annotations as an", "global a, b", "def f():\n    nonlocal a",
  "del a, b[0], c.d", "assert a", "assert a, 'm'", "raise", "raise E", "raise E from c", "return",
  "def f():\n    return a, b", "type A = int", "type A[T: int, *Ts, **P] = Callable[P, tuple[T, *Ts]]",
  "type A = B | C | D", "pass", "while a:\n    break", "while a:\n    continue"]) {
  sameText(text);
}
sameText("from m import (a,\n    b)", "from m import a, b");
console.log("ok");

## PYPRS-07 · Compound statements: conditions, loops, exceptions, context managers and matches

In [ ]:
const chain = first("if a:\n    pass\nelif b:\n    pass\nelif c:\n    pass\nelse:\n    pass");
assert(chain.orelse[0] instanceof S.If && chain.orelse[0].orelse[0] instanceof S.If);
equal(kinds(chain.orelse[0].orelse[0].orelse), ["Pass"]);
assert(first("async def f():\n    async for a in b:\n        pass").body[0] instanceof S.AsyncFor);
const tried = first("try:\n    pass\nexcept (A, B) as e:\n    pass\nexcept C:\n    pass\nexcept:\n    pass\n"
  + "else:\n    pass\nfinally:\n    pass");
equal(tried.handlers.map((h: S.ExceptHandler) => h.name?.spelling ?? null), ["e", null, null]);
assert(tried.handlers[0].type instanceof S.Parenthesized && tried.handlers[2].type === null);
assert(first("try:\n    pass\nexcept A, B:\n    pass").handlers[0].type instanceof S.Tuple);
assert(first("try:\n    pass\nexcept* A:\n    pass") instanceof S.TryStar);
const withStatement = first("with a as b, c:\n    pass");
assert(withStatement.items[0].optional_vars.id.spelling === "b" && withStatement.items[1].optional_vars === null);
assert(first("with (a as b):\n    pass").items[0].optional_vars.id.spelling === "b");
assert(first("async def f():\n    async with a:\n        pass").body[0] instanceof S.AsyncWith);
assert(first("match a, b:\n    case _:\n        pass").subject instanceof S.Tuple);
assert(first("match a,:\n    case _:\n        pass").subject instanceof S.Tuple);
for (const text of ["if a:\n    pass\nelif b:\n    pass\nelse:\n    pass", "while a:\n    pass\nelse:\n    pass",
  "for a in b:\n    pass\nelse:\n    pass", "try:\n    pass\nfinally:\n    pass",
  "try:\n    pass\nexcept* (A, B) as e:\n    pass", "try:\n    pass\nexcept A, B:\n    pass",
  "with a as (b, c), d:\n    pass", "match a:\n    case 1:\n        pass\n    case _:\n        pass",
  "if a:\n    if b:\n        pass\nelse:\n    pass"]) {
  sameText(text);
}
sameText("with (open(a) as b, c):\n    pass", "with open(a) as b, c:\n    pass");
sameText("if a: b = 1; c = 2", "if a:\n    b = 1\n    c = 2");
console.log("ok");

## PYPRS-08 · Definitions: functions, classes, their decorators, parameters and type parameters

In [ ]:
const fn = first("@d\n@e.f(1)\nasync def f[T](a, /, b: int = 1, *c: *Ts, d, e=2, **f: int) -> T:\n    pass");
assert(fn instanceof S.AsyncFunctionDef);
equal(kinds(fn.decorator_list), ["Name", "Call"]);
const args = fn.args as S.Arguments;
const spellings = (list: S.Arg[]) => list.map((a) => (a.arg as S.Identifier).spelling);
equal(spellings(args.posonlyargs), ["a"]);
equal(spellings(args.args), ["b"]);
assert(args.vararg?.arg?.spelling === "c" && args.vararg.annotation instanceof S.Starred);
equal(spellings(args.kwonlyargs), ["d", "e"]);
assert(args.kwonlyargs[1]?.default_value !== null);
assert((args.kwarg?.annotation as S.Name).id?.spelling === "int" && (fn.returns as S.Name).id?.spelling === "T");
equal(spellings(first("def f(*, a): pass").args.kwonlyargs), ["a"]);
const generic = first("class C[T: int, U: (int, str), *Ts, **P](B, metaclass=M, **k):\n    pass");
equal(kinds(generic.type_params), ["TypeVar", "TypeVar", "TypeVarTuple", "ParamSpec"]);
assert(generic.type_params[1].bound instanceof S.Parenthesized);
equal(kinds(generic.bases), ["Name"]);
equal(generic.keywords.map((k: S.Keyword) => k.arg?.spelling ?? null), ["metaclass", null]);
equal(first("class C:\n    pass").bases, []);
for (const text of ["@d\n@e.f(1)\nasync def f[T](a, /, b: int = 1, *c: *Ts, d, e=2, **f: int) -> T:\n    pass",
  "def f(*, a):\n    pass", "def f(a, /):\n    pass", "def f(*a, **b):\n    pass", "@(lambda f: f)\ndef f():\n    pass",
  "class C[T: int, U: (int, str), *Ts, **P](B, metaclass=M, **k):\n    pass", "class C:\n    pass",
  "class C(B):\n    x: int = 0"]) {
  sameText(text);
}
sameText("class C():\n    pass", "class C:\n    pass");
console.log("ok");

## PYPRS-09 · Patterns: values, singletons, sequences, mappings, classes, captures, alternatives and groups

In [ ]:
const pattern = (text: string): any => first(`match x:\n    case ${text}:\n        pass`).cases[0].pattern;

assert(pattern("1") instanceof S.MatchValue && pattern("-1").value.op === "-" && pattern("-1 + 2j").value.op === "+");
assert(pattern("a.b") instanceof S.MatchValue && pattern("a").name.spelling === "a");
assert(pattern("_").name === null && pattern("_").pattern === null);
equal(["None", "True", "False"].map((s) => pattern(s).value), ["None", "True", "False"]);
equal(["[a]", "(a,)", "a, b", "()"].map((s) => pattern(s).delimiters), ["[]", "()", null, "()"]);
assert(pattern("(a)") instanceof S.MatchAs); // a group is its pattern
equal(kinds(pattern("1 | (2 | 3)").patterns), ["MatchValue", "MatchOr"]);
assert(pattern("1 | 2 | 3").patterns.length === 3);
equal(kinds(pattern("-1 | _").patterns), ["MatchValue", "MatchAs"]);
const mapping = pattern("{'k': v, -1: _, **rest}");
equal(kinds(mapping.keys), ["Constant", "UnaryOp"]);
assert(mapping.rest.spelling === "rest");
equal(kinds(pattern("{a.b: x, 1 + 2j: y}").keys), ["Attribute", "BinOp"]);
const cls = pattern("P(1, x=-2, y=_, z=a.b, w=str() as s)");
equal(cls.kwd_attrs.map((a: S.Identifier) => a.spelling), ["x", "y", "z", "w"]);
equal(kinds(cls.kwd_patterns), ["MatchValue", "MatchAs", "MatchValue", "MatchAs"]);
assert(cls.kwd_patterns[3].name.spelling === "s" && cls.kwd_patterns[3].pattern instanceof S.MatchClass);
assert(pattern("[*a, *_]").patterns[1].name === null && pattern("(1 | 2) as s").name.spelling === "s");
for (const text of ["1", "-1", "-1 + 2j", "1.5 - 2j", "'a' 'b'", "b'a'", "a.b.c", "a", "_", "None", "[a, *b]", "(a,)",
  "()", "[]", "{'k': v, -1: _, **rest}", "{}", "{a.b: x, -1 - 2j: y}", "P(1, x=-2, y=_, z=a.b, w=str() as s)", "P()",
  "1 | 2 | 3", "1 | (2 | 3)", "1 | 2 as s", "[a, b] as c", "[(a as b) as c]", "a, b", "a,", "[a | b]"]) {
  sameText(`match x:\n    case ${text}:\n        pass`);
}
sameText("match x:\n    case (a):\n        pass", "match x:\n    case a:\n        pass");
sameText("match x:\n    case (1 | 2) as s:\n        pass", "match x:\n    case 1 | 2 as s:\n        pass");
console.log("ok");

## PYPRS-10 · Comments are kept where statements are listed, on their own lines or trailing; elsewhere they are dropped

In [ ]:
const commented = parse("# lead\nx = 1  # trailing\nif a:  # header\n    pass\n    # end of block\n# between\nelif b:\n"
  + "    pass\nelse:\n    # first\n    pass\ny = (1,  # dropped\n     2)\n");
equal([...F.walk(commented)].filter((c): c is S.Comment => c instanceof S.Comment).map((c) => [c.text, c.trailing]),
  [[" lead", false], [" trailing", true], [" header", true], [" end of block", false], [" between", false],
    [" first", false]]);
equal(kinds((commented.body[3] as S.If).body), ["Comment", "Pass", "Comment", "Comment"]);
const matched = first("match a:\n    case 1:\n        pass\n    # between cases\n    case 2:\n        pass\n");
equal(kinds(matched.cases[0].body), ["Pass", "Comment"]);
const finished = first("try:\n    pass\n# before except\nexcept E:\n    pass\n# before finally\nfinally:\n    pass\n");
equal(kinds(finished.body), ["Pass", "Comment"]);
equal(kinds(finished.handlers[0].body), ["Pass", "Comment"]);
for (const text of ["# lead\nx = 1  # trailing", "if a:  # header\n    pass\nelse:  # else\n    pass",
  "def f():  # header\n    pass  # trailing\n    # own"]) {
  sameText(text);
}
sameText("x = 1 \\\n    + 2", "x = 1 + 2"); // line continuations are layout
console.log("ok");

## PYPRS-11 · Python 3.13 and later, which the pre-pass puts back: lazy imports, dict comprehensions that unpack, and type parameters' defaults

In [ ]:
equal(parse("lazy import json\nlazy from . import x\nimport os\n").body.map((s: any) => s.is_lazy), [true, true, false]);
assert(value("{**d for d in ds}") instanceof S.DictComp && value("{**d for d in ds}").value === null);
assert(value("{**a, **b}") instanceof S.Dict); // a display, not a comprehension
assert(value("{**a for a in b}\n[{**c} for c in d]") instanceof S.DictComp);
const defaults = first("def f[T = int, *Ts = (int,), **P = [str], U: str = 'u'](): pass").type_params;
equal(defaults.map((d: any) => d.default_value.kind().KIND), ["Name", "Parenthesized", "List", "Constant"]);
assert(defaults[3].bound.id.spelling === "str");
assert(first("class C[T = dict[str,\n    int]]: pass").type_params[0].default_value.slice.elts[1].id.spelling === "int");
assert(first("type A[T = int] = list[T]").type_params[0].default_value.id.spelling === "int");
assert(parse("lazy = 1\nlazy(import_)\n").body[0] instanceof S.Assign); // still a name
assert((parse("lazy import x\ny = {**a, 'b': 1}\n").body[1] as S.Assign).value instanceof S.Dict);
for (const text of ["lazy import json", "lazy from . import x", "{**d for d in ds}", "{**a, **b}",
  "def f[T = int, *Ts = (int,), **P = [str]]():\n    pass", "type A[T = int] = list[T]",
  "class C[T: str = 'u']:\n    pass"]) {
  sameText(text);
}
console.log("ok");

## PYPRS-12 · Errors name their line and column, counted in characters

In [ ]:
fails("x = (", "syntax error", 1, 1); // where tree-sitter's error starts
fails("é = (1,\n     é é)", "syntax error", 2, 8);
fails("print 'x'", "unsupported syntax: print_statement, which is Python 2", 1, 1);
fails("exec 'x'", "unsupported syntax: exec_statement, which is Python 2", 1, 1);
fails("a <> b", "unsupported syntax: <>, which is Python 2", 1, 3);
fails("def f((a, b)): pass", "unsupported syntax: tuple_pattern", 1, 7);
fails("def f((a, b)=1): pass", "unsupported syntax: tuple_pattern", 1, 7);
fails("x = a[*b, *(c)]", "syntax error", 1, 11); // tree-sitter-python 0.25 cannot parse it
fails("def f[T = int](): pass", "TypeVar.default_value needs Python 3.13, but this is Python 3.12", 1, 7, Python312);
fails("\n\nx = t'{y}'", "TemplateStr needs Python 3.14, but this is Python 3.12", 3, 5, Python312);
fails("lazy import a", "Import.is_lazy needs Python 3.15, but this is Python 3.14", 1, 1, Python314);
fails("[*a for a in b]", "ListComp of unpacked elements needs Python 3.15, but this is Python 3.14", 1, 1, Python314);
fails("lazy import (", "syntax error", 1, 1);
fails("def f(:", "expected )", 1, 7);
fails("def f[a.b](): pass", "unsupported syntax: attribute", 1, 7);
fails("x = (a as b)", "unsupported syntax: as_pattern", 1, 6);
fails("def f[T = ](): pass", "syntax error", 1, 9);
fails("def f[T = (](): pass", "syntax error", 1, 9);
fails("def f[\n  T = 1 +](): pass", "syntax error", 2, 9); // in a default, which is parsed on its own
fails("\u{1d465} = (1,\n     \u{1d465} \u{1d465})", "syntax error", 2, 8); // beyond UTF-16's first plane, one each
console.log("ok");